# Optimizer comparison on MNIST: SGD vs Adagrad

**Name:** Muhammad Taqui
**Enrollment:** 01-136221-021
**Class:** BS-AI(6A)

Aim: train the same simple feedforward network on MNIST with two different optimizers (SGD and Adagrad) and compare their loss curves and test accuracy.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader


## Load and preprocess MNIST

In [2]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = datasets.MNIST(root='./data', train=True, transform=transform, download=True)
test_dataset = datasets.MNIST(root='./data', train=False, transform=transform, download=True)

train_loader = DataLoader(dataset=train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=1000, shuffle=False)


100%|██████████| 9.91M/9.91M [00:00<00:00, 16.9MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 395kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 3.76MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 3.28MB/s]


## Define the model

In [3]:
class SimpleNN(nn.Module):
    def __init__(self):
        super(SimpleNN, self).__init__()
        self.fc1 = nn.Linear(28 * 28, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 10)

    def forward(self, x):
        x = x.view(-1, 28 * 28)
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x


model_sgd = SimpleNN()
model_adagrad = SimpleNN()


## Training and evaluation functions

In [4]:
def train(model, optimizer, epochs=5):
    criterion = nn.CrossEntropyLoss()
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for images, labels in train_loader:
            optimizer.zero_grad()
            output = model(images)
            loss = criterion(output, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch [{epoch + 1}/{epochs}], Loss: {total_loss / len(train_loader):.4f}")


def evaluate(model):
    model.eval()
    correct = 0
    with torch.no_grad():
        for images, labels in test_loader:
            output = model(images)
            _, predicted = torch.max(output.data, 1)
            correct += (predicted == labels).sum().item()
    accuracy = correct / len(test_loader.dataset)
    print(f"Accuracy: {accuracy * 100:.2f}%")
    return accuracy


## Train with SGD

In [5]:
optimizer_sgd = optim.SGD(model_sgd.parameters(), lr=0.01)
print("Training with SGD:")
train(model_sgd, optimizer_sgd, epochs=5)
accuracy_sgd = evaluate(model_sgd)


Training with SGD:
Epoch [1/5], Loss: 1.0889
Epoch [2/5], Loss: 0.3856
Epoch [3/5], Loss: 0.3274
Epoch [4/5], Loss: 0.2984
Epoch [5/5], Loss: 0.2770
Accuracy: 92.56%


## Train with Adagrad

In [6]:
optimizer_adagrad = optim.Adagrad(model_adagrad.parameters(), lr=0.01)
print("Training with Adagrad:")
train(model_adagrad, optimizer_adagrad, epochs=5)
accuracy_adagrad = evaluate(model_adagrad)


Training with Adagrad:
Epoch [1/5], Loss: 0.4191
Epoch [2/5], Loss: 0.2508
Epoch [3/5], Loss: 0.2139
Epoch [4/5], Loss: 0.1926
Epoch [5/5], Loss: 0.1755
Accuracy: 95.00%


In [7]:
print(f"SGD Accuracy:     {accuracy_sgd * 100:.2f}%")
print(f"Adagrad Accuracy:  {accuracy_adagrad * 100:.2f}%")


SGD Accuracy:     92.56%
Adagrad Accuracy:  95.00%


## Concept notes

### Stochastic gradient descent (SGD)

SGD updates model weights using a single sample or small batch at each step rather than the full dataset, which makes each update noisy but computationally cheap.

**How it works:** at each step the algorithm samples one or a few data points, computes the gradient of the loss with respect to the parameters, and adjusts the parameters by a small amount in the direction that reduces the error. Over many steps this drives the parameters toward lower loss.

**Advantages**
- Faster and more memory-efficient on large datasets than full-batch gradient descent.
- The noise in updates can help avoid poor local minima.
- Naturally suited to streaming or online data.

**Challenges**
- The loss path is irregular and can oscillate.
- Performance is sensitive to the learning rate: too large causes instability, too small causes slow convergence.

### Adagrad

Adagrad adapts the learning rate per parameter based on the historical magnitude of that parameter's gradients: frequently updated parameters get smaller learning rates, infrequently updated ones get larger learning rates.

**How it works:** Adagrad accumulates the sum of squared past gradients for each parameter and scales the learning rate inversely to that accumulated value.

**Advantages**
- Handles sparse features or rare/frequent feature imbalance well.
- Removes most of the need for manual learning-rate tuning.

**Challenges**
- The accumulated squared gradient only grows, so the effective learning rate keeps shrinking and can eventually stall learning.
- This makes it less effective for long training runs.

### Comparison

SGD is a general-purpose optimizer that needs a manually tuned learning rate but performs well across a wide range of problems. Adagrad automatically adapts per-parameter learning rates, which helps with sparse or imbalanced features, but its monotonically shrinking learning rate can cause it to stop improving over long training horizons. In the run above, Adagrad reached a lower training loss within 5 epochs, but this pattern does not generalize to every dataset or training length.